# Migration of Dept_pkg
**Source:** ODI SQL Task Set
**Target:** Databricks Spark SQL Jupyter Notebook
**Conversion Timestamp:** 2023-10-27

In [ ]:
dbutils.widgets.text("v_lst_dt", "")
dbutils.widgets.text("ODI_SESS_NO", "af20e4bd-d7e8-48f7-9e6e-163089625fc1")

### ETL Parameters (SCEN_TASK_NO 1)

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_last_run_param AS
SELECT COALESCE(
    (SELECT date_format(last_run_dt, 'dd-MM-yy') FROM workspace.odi_trg.log_table1 WHERE pkg_name = 'Dept_pkg'),
    date_format(current_timestamp(), 'dd-MM-yy')
) AS last_run_dt;

### Update Log Table (SCEN_TASK_NO 2)

In [ ]:
%sql
UPDATE workspace.odi_trg.log_table1 
SET last_run_dt = current_timestamp() 
WHERE pkg_name = 'Dept_pkg';

### Staging Table Preparation (SCEN_TASK_NO 30, 40)

In [ ]:
%sql
DROP TABLE IF EXISTS workspace.odi_trg.c_0filter;

CREATE TABLE workspace.odi_trg.c_0filter
(
    DEPARTMENT_ID   BIGINT,
    DEPARTMENT_NAME STRING,
    MANAGER_ID      BIGINT,
    LOCATION_ID     BIGINT,
    LAST_UPD_DT     TIMESTAMP
)
USING DELTA;

### Extract into Staging (SCEN_TASK_NO 50)

In [ ]:
%sql
INSERT INTO workspace.odi_trg.c_0filter
(
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT
)
SELECT
    SRC_DEPARTMENTS.DEPARTMENT_ID,
    SRC_DEPARTMENTS.DEPARTMENT_NAME,
    SRC_DEPARTMENTS.MANAGER_ID,
    SRC_DEPARTMENTS.LOCATION_ID,
    SRC_DEPARTMENTS.LAST_UPD_DT
FROM workspace.odi_src.src_departments AS SRC_DEPARTMENTS
WHERE SRC_DEPARTMENTS.LAST_UPD_DT >= to_date('${v_lst_dt}', 'dd-MM-yy');

### Optimize Staging (SCEN_TASK_NO 60)

In [ ]:
%sql
OPTIMIZE workspace.odi_trg.c_0filter;

### Flow Table Preparation (SCEN_TASK_NO 80, 90)

In [ ]:
%sql
DROP TABLE IF EXISTS workspace.odi_trg.i_trg_departments_flow;

CREATE TABLE workspace.odi_trg.i_trg_departments_flow
(
    DEPARTMENT_ID       BIGINT,
    DEPARTMENT_NAME     STRING,
    MANAGER_ID          BIGINT,
    LOCATION_ID         BIGINT,
    LAST_UPD_DT         TIMESTAMP,
    IND_UPDATE          STRING
)
USING DELTA;

### Load Integration Flow (SCEN_TASK_NO 100)

In [ ]:
%sql
INSERT INTO workspace.odi_trg.i_trg_departments_flow
(
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT,
    IND_UPDATE
)
SELECT 
    S.DEPARTMENT_ID,
    S.DEPARTMENT_NAME,
    S.MANAGER_ID,
    S.LOCATION_ID,
    S.LAST_UPD_DT,
    'I' AS IND_UPDATE
FROM workspace.odi_trg.c_0filter AS S
WHERE NOT EXISTS (
    SELECT 1 FROM workspace.odi_trg.trg_departments AS T
    WHERE T.DEPARTMENT_ID = S.DEPARTMENT_ID 
      AND (T.DEPARTMENT_NAME <=> S.DEPARTMENT_NAME)
      AND (T.MANAGER_ID <=> S.MANAGER_ID)
      AND (T.LOCATION_ID <=> S.LOCATION_ID)
      AND (T.LAST_UPD_DT <=> S.LAST_UPD_DT)
);

### Flow Table Optimization (SCEN_TASK_NO 110, 120)

In [ ]:
%sql
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.odi_trg.i_trg_departments_flow ZORDER BY (DEPARTMENT_ID);

### Audit Tables Preparation (SCEN_TASK_NO 130, 140, 150, 160)

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS workspace.odi_trg.snp_check_tab
(
    CATALOG_NAME    STRING,
    SCHEMA_NAME     STRING,
    RESOURCE_NAME   STRING,
    FULL_RES_NAME   STRING,
    ERR_TYPE        STRING,
    ERR_MESS        STRING,
    CHECK_DATE      TIMESTAMP,
    ORIGIN          STRING,
    CONS_NAME       STRING,
    CONS_TYPE       STRING,
    ERR_COUNT       BIGINT
) USING DELTA;

DELETE FROM workspace.odi_trg.snp_check_tab
WHERE SCHEMA_NAME = 'ODI_TRG'
  AND ORIGIN = '(7)test.Dept_MAp'
  AND ERR_TYPE = 'F';

CREATE TABLE IF NOT EXISTS workspace.odi_trg.e_trg_departments
(
    ODI_ROW_ID      STRING,
    ODI_ERR_TYPE    STRING, 
    ODI_ERR_MESS    STRING,
    ODI_CHECK_DATE  TIMESTAMP, 
    DEPARTMENT_ID   BIGINT,
    DEPARTMENT_NAME STRING,
    MANAGER_ID      BIGINT,
    LOCATION_ID     BIGINT,
    LAST_UPD_DT     TIMESTAMP,
    ODI_ORIGIN      STRING,
    ODI_CONS_NAME   STRING,
    ODI_CONS_TYPE   STRING,
    ODI_PK          STRING,
    ODI_SESS_NO     STRING
) USING DELTA;

DELETE FROM workspace.odi_trg.e_trg_departments
WHERE (ODI_ERR_TYPE = 'F' AND ODI_ORIGIN = '(7)test.Dept_MAp');

### Constraints Verification (SCEN_TASK_NO 180, 190)

In [ ]:
%sql
INSERT INTO workspace.odi_trg.e_trg_departments
(
    ODI_PK,
    ODI_SESS_NO,
    ODI_ROW_ID,
    ODI_ERR_TYPE,
    ODI_ERR_MESS,
    ODI_ORIGIN,
    ODI_CHECK_DATE,
    ODI_CONS_NAME,
    ODI_CONS_TYPE,
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT
)
SELECT
    uuid(),
    '${ODI_SESS_NO}',
    CAST(monotonically_increasing_id() AS STRING),
    'F',
    'ODI-15064: The primary key PK_department_id is not unique.',
    '(7)test.Dept_MAp',
    current_timestamp(),
    'PK_department_id',
    'PK',
    T.DEPARTMENT_ID,
    T.DEPARTMENT_NAME,
    T.MANAGER_ID,
    T.LOCATION_ID,
    T.LAST_UPD_DT
FROM workspace.odi_trg.i_trg_departments_flow AS T
WHERE EXISTS (
    SELECT 1 FROM workspace.odi_trg.i_trg_departments_flow AS SUB
    WHERE SUB.DEPARTMENT_ID = T.DEPARTMENT_ID
    GROUP BY SUB.DEPARTMENT_ID
    HAVING COUNT(1) > 1
);

INSERT INTO workspace.odi_trg.e_trg_departments
(
    ODI_PK,
    ODI_SESS_NO,
    ODI_ROW_ID,
    ODI_ERR_TYPE,
    ODI_ERR_MESS,
    ODI_CHECK_DATE,
    ODI_ORIGIN,
    ODI_CONS_NAME,
    ODI_CONS_TYPE,
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT
)
SELECT
    uuid(),
    '${ODI_SESS_NO}',
    CAST(monotonically_increasing_id() AS STRING),
    'F',
    'ODI-15066: The column DEPARTMENT_NAME cannot be null.',
    current_timestamp(),
    '(7)test.Dept_MAp',
    'DEPARTMENT_NAME',
    'NN',
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT
FROM workspace.odi_trg.i_trg_departments_flow
WHERE DEPARTMENT_NAME IS NULL;

### Error Clean-up and Logging (SCEN_TASK_NO 210, 220)

In [ ]:
%sql
DELETE FROM workspace.odi_trg.i_trg_departments_flow
WHERE DEPARTMENT_ID IN (
    SELECT E.DEPARTMENT_ID 
    FROM workspace.odi_trg.e_trg_departments AS E
    WHERE E.ODI_SESS_NO = '${ODI_SESS_NO}'
);

INSERT INTO workspace.odi_trg.snp_check_tab
(
    SCHEMA_NAME,
    RESOURCE_NAME,
    FULL_RES_NAME,
    ERR_TYPE,
    ERR_MESS,
    CHECK_DATE,
    ORIGIN,
    CONS_NAME,
    CONS_TYPE,
    ERR_COUNT
)
SELECT
    'ODI_TRG',
    'TRG_DEPARTMENTS',
    'workspace.odi_trg.trg_departments',
    E.ODI_ERR_TYPE,
    E.ODI_ERR_MESS,
    E.ODI_CHECK_DATE,
    E.ODI_ORIGIN,
    E.ODI_CONS_NAME,
    E.ODI_CONS_TYPE,
    COUNT(1)
FROM workspace.odi_trg.e_trg_departments AS E
WHERE E.ODI_ERR_TYPE = 'F'
  AND E.ODI_ORIGIN = '(7)test.Dept_MAp'
GROUP BY 
    E.ODI_ERR_TYPE, E.ODI_ERR_MESS, E.ODI_CHECK_DATE, E.ODI_ORIGIN, E.ODI_CONS_NAME, E.ODI_CONS_TYPE;

### Flag Records for Update (SCEN_TASK_NO 230)

In [ ]:
%sql
MERGE INTO workspace.odi_trg.i_trg_departments_flow AS T
USING (
    SELECT DEPARTMENT_ID
    FROM workspace.odi_trg.trg_departments
) AS S
ON T.DEPARTMENT_ID = S.DEPARTMENT_ID
WHEN MATCHED THEN UPDATE SET T.IND_UPDATE = 'U';

### Merge into Target (SCEN_TASK_NO 270, 280)

In [ ]:
%sql
MERGE INTO workspace.odi_trg.trg_departments AS T
USING workspace.odi_trg.i_trg_departments_flow AS S
ON T.DEPARTMENT_ID = S.DEPARTMENT_ID
WHEN MATCHED AND S.IND_UPDATE = 'U' THEN UPDATE SET
    T.DEPARTMENT_NAME = S.DEPARTMENT_NAME,
    T.MANAGER_ID      = S.MANAGER_ID,
    T.LOCATION_ID     = S.LOCATION_ID,
    T.LAST_UPD_DT     = S.LAST_UPD_DT
WHEN NOT MATCHED AND S.IND_UPDATE = 'I' THEN INSERT (
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT
) VALUES (
    S.DEPARTMENT_ID,
    S.DEPARTMENT_NAME,
    S.MANAGER_ID,
    S.LOCATION_ID,
    S.LAST_UPD_DT
);

### Final Cleanup (SCEN_TASK_NO 340, 360)

In [ ]:
%sql
DROP TABLE IF EXISTS workspace.odi_trg.c_0filter;

DROP TABLE IF EXISTS workspace.odi_trg.i_trg_departments_flow;